In [7]:
import os

print(os.listdir("/lakehouse/default/Files/Test_files"))

from notebookutils import mssparkutils

for f in mssparkutils.fs.ls("Files/Test_files"):
    print(f)

raw_header = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Test_files/po_raw_header.csv")
)

display(raw_header)

StatementMeta(, be9a3af3-0b02-4a43-a9d7-a7f6a29340c6, 9, Finished, Available, Finished, False)

['erp_item_hold.csv', 'erp_item_master.csv', 'erp_sales_order_status.csv', 'erp_uom_conversion.csv', 'po_raw_header.csv', 'po_raw_lines.csv']
FileInfo(path=abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/erp_item_hold.csv, name=erp_item_hold.csv, size=87)
FileInfo(path=abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/erp_item_master.csv, name=erp_item_master.csv, size=266)
FileInfo(path=abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/erp_sales_order_status.csv, name=erp_sales_order_status.csv, size=181)
FileInfo(path=abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/erp_uom_conversion.csv, name=erp_uom_conversion.csv, size=60)
FileInfo(path=abfss://c987b25b-3

SynapseWidget(Synapse.DataFrame, 72dfc135-73ae-4982-8ff0-5c16441e920b)

In [12]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------------------------
# LOAD CSV FILES FROM EXACT ABFSS PATHS
# ---------------------------------
raw_header = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/po_raw_header.csv")
)

raw_lines = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/po_raw_lines.csv")
)

sales_status = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/erp_sales_order_status.csv")
)

item_master = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/erp_item_master.csv")
)

uom_conv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/erp_uom_conversion.csv")
)

item_hold = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("abfss://c987b25b-3200-4853-8b17-813784a3657e@onelake.dfs.fabric.microsoft.com/7c4700cf-8740-4afb-9dbf-0a7051a84827/Files/Test_files/erp_item_hold.csv")
)

# ---------------------------------
# CLEAN / CAST TYPES
# ---------------------------------
raw_header = (
    raw_header
    .select(
        F.col("document_id").cast("string").alias("document_id"),
        F.col("customer_id").cast("string").alias("customer_id"),
        F.col("sales_order_no").cast("string").alias("sales_order_no"),
        F.col("doc_total").cast("double").alias("doc_total"),
        F.col("source_file_name").cast("string").alias("source_file_name")
    )
)

raw_lines = (
    raw_lines
    .select(
        F.col("document_id").cast("string").alias("document_id"),
        F.col("line_no").cast("int").alias("line_no"),
        F.col("item_extracted").cast("string").alias("item_extracted"),
        F.col("description_extracted").cast("string").alias("description_extracted"),
        F.col("qty_extracted").cast("double").alias("qty_extracted"),
        F.col("uom_extracted").cast("string").alias("uom_extracted"),
        F.col("unit_price_extracted").cast("double").alias("unit_price_extracted")
    )
)

sales_status = (
    sales_status
    .select(
        F.col("sales_order_no").cast("string").alias("sales_order_no"),
        F.col("sales_order_status").cast("string").alias("sales_order_status")
    )
)

item_master = (
    item_master
    .select(
        F.col("item_no").cast("string").alias("item_no"),
        F.col("item_description").cast("string").alias("item_description"),
        F.col("standard_uom").cast("string").alias("standard_uom"),
        F.col("min_order_qty").cast("double").alias("min_order_qty")
    )
)

uom_conv = (
    uom_conv
    .select(
        F.col("item_no").cast("string").alias("item_no_uom"),
        F.col("from_uom").cast("string").alias("from_uom_uom"),
        F.col("to_uom").cast("string").alias("to_uom_uom"),
        F.col("conversion_factor").cast("double").alias("conversion_factor")
    )
)

item_hold = (
    item_hold
    .select(
        F.col("item_no").cast("string").alias("item_no_hold"),
        F.col("site_code").cast("string").alias("site_code"),
        F.col("item_hold_flag").cast("boolean").alias("item_hold_flag"),
        F.col("hold_reason").cast("string").alias("hold_reason")
    )
)

# ---------------------------------
# NORMALIZE STRINGS
# ---------------------------------
lines = (
    raw_lines
    .withColumn("item_extracted_norm", F.upper(F.trim(F.col("item_extracted"))))
    .withColumn("description_extracted_norm", F.upper(F.trim(F.col("description_extracted"))))
    .withColumn("uom_extracted_norm", F.upper(F.trim(F.col("uom_extracted"))))
)

items = (
    item_master
    .withColumn("item_no_norm", F.upper(F.trim(F.col("item_no"))))
    .withColumn("item_description_norm", F.upper(F.trim(F.col("item_description"))))
)

# ---------------------------------
# EXACT MATCH ON ITEM NUMBER
# ---------------------------------
exact_item_match = (
    lines.alias("l")
    .join(
        items.alias("i"),
        F.col("l.item_extracted_norm") == F.col("i.item_no_norm"),
        "left"
    )
    .select(
        F.col("l.document_id"),
        F.col("l.line_no"),
        F.col("l.item_extracted"),
        F.col("l.description_extracted"),
        F.col("l.qty_extracted"),
        F.col("l.uom_extracted"),
        F.col("l.unit_price_extracted"),
        F.col("l.item_extracted_norm"),
        F.col("l.description_extracted_norm"),
        F.col("l.uom_extracted_norm"),
        F.col("i.item_no").alias("matched_item_no_exact"),
        F.col("i.item_description").alias("matched_item_desc_exact"),
        F.col("i.standard_uom").alias("standard_uom_exact"),
        F.col("i.min_order_qty").alias("min_order_qty_exact")
    )
)

# ---------------------------------
# CLOSE MATCH USING LEVENSHTEIN
# ---------------------------------
candidate_matches = (
    lines.alias("l")
    .crossJoin(items.alias("i"))
    .withColumn(
        "desc_distance",
        F.levenshtein(F.col("l.description_extracted_norm"), F.col("i.item_description_norm"))
    )
    .withColumn(
        "item_distance",
        F.levenshtein(F.col("l.item_extracted_norm"), F.col("i.item_no_norm"))
    )
    .withColumn(
        "match_score",
        F.col("desc_distance") + F.col("item_distance")
    )
)

w = Window.partitionBy("document_id", "line_no").orderBy(F.col("match_score").asc())

best_candidate = (
    candidate_matches
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        F.col("document_id"),
        F.col("line_no"),
        F.col("item_no").alias("suggested_item_no"),
        F.col("item_description").alias("suggested_item_description"),
        F.col("standard_uom").alias("suggested_standard_uom"),
        F.col("min_order_qty").alias("suggested_min_order_qty"),
        F.col("match_score")
    )
)

# ---------------------------------
# MERGE EXACT + CLOSE MATCH RESULTS
# ---------------------------------
enriched = (
    exact_item_match.alias("e")
    .join(best_candidate.alias("b"), ["document_id", "line_no"], "left")
    .join(raw_header.alias("h"), ["document_id"], "left")
    .join(sales_status.alias("s"), F.col("h.sales_order_no") == F.col("s.sales_order_no"), "left")
    .select(
        F.col("document_id"),
        F.col("line_no"),
        F.col("e.item_extracted"),
        F.col("e.description_extracted"),
        F.col("e.qty_extracted"),
        F.col("e.uom_extracted"),
        F.col("e.unit_price_extracted"),
        F.col("e.item_extracted_norm"),
        F.col("e.description_extracted_norm"),
        F.col("e.uom_extracted_norm"),
        F.col("matched_item_no_exact"),
        F.col("matched_item_desc_exact"),
        F.col("standard_uom_exact"),
        F.col("min_order_qty_exact"),
        F.col("suggested_item_no"),
        F.col("suggested_item_description"),
        F.col("suggested_standard_uom"),
        F.col("suggested_min_order_qty"),
        F.col("match_score"),
        F.col("h.customer_id"),
        F.col("h.sales_order_no"),
        F.col("h.doc_total"),
        F.col("h.source_file_name"),
        F.col("s.sales_order_status")
    )
    .withColumn(
        "matched_item_no",
        F.when(F.col("matched_item_no_exact").isNotNull(), F.col("matched_item_no_exact"))
         .when(F.col("match_score") <= 10, F.col("suggested_item_no"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "matched_item_desc",
        F.when(F.col("matched_item_desc_exact").isNotNull(), F.col("matched_item_desc_exact"))
         .when(F.col("match_score") <= 10, F.col("suggested_item_description"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "match_type",
        F.when(F.col("matched_item_no_exact").isNotNull(), F.lit("EXACT_MATCH"))
         .when(
             (F.col("matched_item_no_exact").isNull()) & (F.col("match_score") <= 10),
             F.lit("CLOSE_MATCH")
         )
         .otherwise(F.lit("NO_MATCH"))
    )
    .withColumn(
        "standard_uom",
        F.when(F.col("matched_item_no_exact").isNotNull(), F.col("standard_uom_exact"))
         .otherwise(F.col("suggested_standard_uom"))
    )
    .withColumn(
        "min_order_qty",
        F.when(F.col("matched_item_no_exact").isNotNull(), F.col("min_order_qty_exact"))
         .otherwise(F.col("suggested_min_order_qty"))
    )
)

# ---------------------------------
# UOM VALIDATION + CONVERSION
# ---------------------------------
enriched = (
    enriched.alias("e")
    .join(
        uom_conv.alias("u"),
        (F.col("e.matched_item_no") == F.col("u.item_no_uom")) &
        (F.col("e.uom_extracted_norm") == F.col("u.from_uom_uom")) &
        (F.col("e.standard_uom") == F.col("u.to_uom_uom")),
        "left"
    )
    .select(
        F.col("e.*"),
        F.col("u.conversion_factor")
    )
    .withColumn(
        "uom_validation_status",
        F.when(F.col("matched_item_no").isNull(), F.lit("SKIP"))
         .when(F.col("uom_extracted_norm") == F.col("standard_uom"), F.lit("PASS"))
         .when(F.col("conversion_factor").isNotNull(), F.lit("PASS_WITH_CONVERSION"))
         .otherwise(F.lit("FAIL"))
    )
    .withColumn(
        "validated_qty",
        F.when(F.col("uom_extracted_norm") == F.col("standard_uom"), F.col("qty_extracted"))
         .when(F.col("conversion_factor").isNotNull(), F.col("qty_extracted") * F.col("conversion_factor"))
         .otherwise(F.col("qty_extracted"))
    )
)

# ---------------------------------
# MOQ VALIDATION
# ---------------------------------
enriched = enriched.withColumn(
    "moq_validation_status",
    F.when(F.col("matched_item_no").isNull(), F.lit("SKIP"))
     .when(F.col("validated_qty") >= F.col("min_order_qty"), F.lit("PASS"))
     .otherwise(F.lit("FAIL"))
)

# ---------------------------------
# ITEM HOLD VALIDATION
# ---------------------------------
enriched = (
    enriched.alias("e")
    .join(
        item_hold.alias("ih"),
        F.col("e.matched_item_no") == F.col("ih.item_no_hold"),
        "left"
    )
    .select(
        F.col("e.*"),
        F.col("ih.item_hold_flag"),
        F.col("ih.hold_reason"),
        F.col("ih.site_code")
    )
    .withColumn(
        "item_hold_validation_status",
        F.when(F.coalesce(F.col("item_hold_flag"), F.lit(False)) == True, F.lit("FAIL"))
         .otherwise(F.lit("PASS"))
    )
)

# ---------------------------------
# SALES ORDER STATUS VALIDATION
# ---------------------------------
enriched = enriched.withColumn(
    "order_status_validation_status",
    F.when(F.col("sales_order_status") == "Journal", F.lit("PASS"))
     .otherwise(F.lit("FAIL"))
)

# ---------------------------------
# UNIT PRICE VALIDATION
# ---------------------------------
enriched = enriched.withColumn(
    "unit_price_validation_status",
    F.when(F.col("unit_price_extracted").isNull(), F.lit("WARN_MISSING"))
     .otherwise(F.lit("PASS"))
)

# ---------------------------------
# LINE TOTAL + DOCUMENT TOTAL VALIDATION
# ---------------------------------
enriched = enriched.withColumn(
    "line_total_calc",
    F.when(F.col("unit_price_extracted").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("validated_qty") * F.col("unit_price_extracted"))
)

doc_totals = (
    enriched.groupBy("document_id")
    .agg(F.sum("line_total_calc").alias("calculated_doc_total"))
)

enriched = (
    enriched.alias("e")
    .join(doc_totals.alias("d"), "document_id", "left")
    .select(
        F.col("e.*"),
        F.col("d.calculated_doc_total")
    )
    .withColumn(
        "total_validation_status",
        F.when(F.col("calculated_doc_total").isNull(), F.lit("WARN_UNABLE_TO_VALIDATE"))
         .when(F.abs(F.col("calculated_doc_total") - F.col("doc_total")) <= 0.01, F.lit("PASS"))
         .otherwise(F.lit("FAIL"))
    )
)

# ---------------------------------
# FINAL LINE STATUS
# ---------------------------------
enriched = (
    enriched
    .withColumn(
        "blocking_error_count",
        F.expr("""
            int(order_status_validation_status = 'FAIL') +
            int(match_type = 'NO_MATCH') +
            int(uom_validation_status = 'FAIL') +
            int(moq_validation_status = 'FAIL') +
            int(item_hold_validation_status = 'FAIL') +
            int(total_validation_status = 'FAIL')
        """)
    )
    .withColumn(
        "final_line_status",
        F.when(F.col("blocking_error_count") > 0, F.lit("BLOCKED"))
         .when(
             (F.col("match_type") == "CLOSE_MATCH") |
             (F.col("unit_price_validation_status") == "WARN_MISSING"),
             F.lit("REQUIRES_REVIEW")
         )
         .otherwise(F.lit("READY"))
    )
    .withColumn("sales_origin", F.lit("SalesAgent"))
)

# ---------------------------------
# SAVE OUTPUTS TO TABLES
# ---------------------------------
(
    enriched.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("po_enriched_lines")
)

rules = [
    ("ORDER_STATUS", "order_status_validation_status"),
    ("UNIT_PRICE", "unit_price_validation_status"),
    ("TOTAL_AMOUNT", "total_validation_status"),
    ("ITEM_MATCH", "match_type"),
    ("UOM", "uom_validation_status"),
    ("MOQ", "moq_validation_status"),
    ("ITEM_HOLD", "item_hold_validation_status"),
]

validation_dfs = []
for rule_name, col_name in rules:
    validation_dfs.append(
        enriched.select(
            "document_id",
            "line_no",
            F.lit(rule_name).alias("rule_name"),
            F.col(col_name).alias("rule_result")
        )
    )

validation_df = validation_dfs[0]
for df in validation_dfs[1:]:
    validation_df = validation_df.unionByName(df)

(
    validation_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("po_validation_results")
)

print("Notebook 2 complete.")

print("Enriched line results")
display(enriched.orderBy("document_id", "line_no"))

print("Validation results")
display(validation_df.orderBy("document_id", "line_no", "rule_name"))

StatementMeta(, be9a3af3-0b02-4a43-a9d7-a7f6a29340c6, 14, Finished, Available, Finished, False)

Notebook 2 complete.
Enriched line results


SynapseWidget(Synapse.DataFrame, d4327a58-82fa-4fa5-b5ee-58fc43ca6da3)

Validation results


SynapseWidget(Synapse.DataFrame, 90903439-08f5-48d8-a42e-4bfa9b2c9108)